# Codificación One-Hot

La codificacion One Hot en contexto de la vectorización de textos es una tecnica que mapea palabras a tensores, $N$ dimencionales ($N=|vocabulario|$) donde todos los valores son 0 a exepcion de indice i que corresponda a la palabra codificada en el vocabulario. Tambien se aplica a nivel de caracteres.

Ejemplo de juguete de codificación One-Hot a nivel de palabras:

In [ ]:
import numpy as np

samples = ['The cat sat on the mat.', 'The dog ate my homework.']

# Tokenizar
token_index = {}
for sample in samples:
    for word in sample.split(): # Dividir palabras por espacios en blanco, en un caso real ademas hay que considerar distintos tipos signos de puntuación.
        if word not in token_index:
            token_index[word] = len(token_index) + 1    # Cada palabra aparece una vez como valor el orden de aparicion.

max_length = 10 # Solo se consideran las primeras 10 palabras de cada muestra (en este caso las cubre al 100%)

# Vectorizar
results = np.zeros(
    shape=(len(samples),                # muestras
    max_length,                         # 10 palabras
    max(token_index.values()) + 1))     # vector de tamaño 11 por palabra (el 0 se salta aproposito)
for i, sample in enumerate(samples):
    for j, word in list(enumerate(sample.split()))[:max_length]:
        index = token_index.get(word)
        results[i, j, index] = 1.

print(results[0])

[[0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]]


Ejemplo de juguete a nivel de caracteres:

In [ ]:
import string

samples = ['The cat sat on the mat.', 'The dog ate my homework.']

characters = string.printable   # Todos los caracteres ASCII imprimibles

# Tokenizado
token_index = dict(zip(characters, range(1, len(characters) + 1)))

max_length = 50 # Maxima cantidad de caracteres por muestra

# Vectorizado
results = np.zeros((len(samples), max_length, max(token_index.values()) + 1)) # Tamaño (x,x,101) (100 posibles tokens + 0)
for i, sample in enumerate(samples):
    for j, character in enumerate(sample):
        index = token_index.get(character)
        results[i, j, index] = 1.

print(results[0])

[[0. 1. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 ...
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]]


Ejemplo de codificación One-Hot con utilidades de keras:

In [2]:
from tensorflow.keras.preprocessing.text import Tokenizer

samples = ['The cat sat on the mat.', 'The dog ate my homework.']

tokenizer = Tokenizer(num_words=1000)   # Toma en cuenta solo las 1000 palabras mas comunes del conjunto de datos.
tokenizer.fit_on_texts(samples)

sequences = tokenizer.texts_to_sequences(samples)   # Esto convierte las muestras en un arreglo de arreglos de indices

one_hot_result = tokenizer.texts_to_matrix(samples, mode='binary')  # Esto directamente da la representacion one_hot representada

word_index = tokenizer.word_index
print('Found %s unique tokens.' % len(word_index))

Found 9 unique tokens.


El uso de `Tokenizer` evita tener que implementar la logica de tokenizado de forma manual, ademas de que es una funcion implementada por la propia libreria que asegura cierta metadologia y ventajas extra como tener en cuenta las $N$ palabras mas populares.

Otros metodos como `layer.TextVectorization` optan por agregar una capa extra delante del modelo que se encarga de procesar realizar el vectorizado (tambien se pueden usar fuera del modelo de forma similar a `Tokenizer`).

Otra opcion para la vectorizacion de muestras es *one_hot hashing trick*, consiste en hashear cada palabra en un arreglo de tamaño fijo. Es util cuando la cantidad de tokens unicos es muy grande. Logra ahorrar memoria ya que no necesita el diccionario de vocabulario (esta implicito en la funcion de hash). Es suceptible a *hash collisions* (el modelo no puede diferenciar entre dos palabras hasheadas al mismo vector), se puede tratar haciendo el espacio de hash mucho mas grande que el numero de tokens unicos.

Ejemplo de codificación *one_hot hashing trick*:

In [7]:
import numpy as np

samples = ['The cat sat on the mat.', 'The dog ate my homework.']

dimensionality = 1000
max_length = 10

results = np.zeros((len(samples), max_length, dimensionality)) # Para cada ejemplo se consideran max 10 palabras codificadas en vectores de |v| = 1000
for i, sample in enumerate(samples):
    for j, word in list(enumerate(sample.split()))[:max_length]:
        index = abs(hash(word)) % dimensionality
        results[i,j,index] = 1

print(results)

[[[0. 0. 0. ... 0. 0. 0.]
  [0. 0. 0. ... 0. 0. 0.]
  [0. 0. 0. ... 0. 0. 0.]
  ...
  [0. 0. 0. ... 0. 0. 0.]
  [0. 0. 0. ... 0. 0. 0.]
  [0. 0. 0. ... 0. 0. 0.]]

 [[0. 0. 0. ... 0. 0. 0.]
  [0. 0. 0. ... 0. 0. 0.]
  [0. 0. 0. ... 0. 0. 0.]
  ...
  [0. 0. 0. ... 0. 0. 0.]
  [0. 0. 0. ... 0. 0. 0.]
  [0. 0. 0. ... 0. 0. 0.]]]


El formato final de las palabras es el mismo, la ventaja de usar una funcion de hashin radica en ahorrar memoria RAM al no tener que tener cargado el diccionario de vocabulario, haciendo que una funcion de hash (usualmente ligera) tome su lugar. 

En el ejemplo no hay coliciones ya que el vocabulario es muy pequeño (9 palabras) pero si este aumenta a un valor cercano o superior a 1000 (espacio de hash actual) habria que incrementar el espacio de hash. Lo que implica valor de `dimensionality` mas grande y el ajuste correspondiente a la funcion de hash. De otra forma aumentara la cantidad de colisiones.